# Dynp Parameter Grid: Precision/Recall/F1 vs. cost, n_bkps, min_size (red/snr3)

Single-method, single-condition diagnostic: fix `ruptures` algorithm to **Dynp** (exact dynamic
programming), run it on the **red/snr3** noise condition only
(`noise_sensitivity_data/noise_red_snr3.npz`, 20 seeds), and grid-search `cost` x `n_bkps` x
`min_size` for all 9 `ruptures` cost functions (`l1`, `l2`, `rbf`, `normal`, `ar`, `linear`,
`rank`, `clinear`, `mahalanobis`). Mirrors `pelt_grid_red_snr3.ipynb` /
`binseg_grid_red_snr3.ipynb` / `bottomup_grid_red_snr3.ipynb` -- same data, same ground truth,
same overall methodology, with the changes below.

**No `pen` -- `n_bkps` instead.** Dynp is exact: it has no penalty term to trade fit quality
against segment count, so `ruptures_transitions.py` requires `n_bkps` (an exact, fixed number of
breakpoints) and rejects `pen` outright for this algorithm. Every place `pen` appeared in the
other notebooks (grid axis, sweep, surface plot, conclusion columns) is `n_bkps` here instead.

**Ground truth transition counts (informs the `n_bkps` grid below).** Computed directly from
`noise_red_snr3.npz`'s `state_true` for all 20 seeds: min=35, max=60, mean=44.85. `N_BKPS_GRID`
below spans below/above this range (10-100) so the winner isn't forced onto an edge by an
under-ranged grid.

**Important correction vs. the pen-based notebooks: `ar`/`linear` will NOT show zero detections
here.** In Pelt/Binseg/BottomUp/Window, `ar` and `linear` came back all-zero because the
*penalty* never justified placing any breakpoint for those cost functions on this data. Dynp has
no penalty -- given `n_bkps`, it is *forced* to place exactly that many breakpoints regardless of
whether the cost function actually discriminates real transitions from noise. So `ar`/`linear`
will still emit `n_bkps` breakpoints every time; expect those breakpoints to land close to
uninformative/arbitrary locations (poor precision/recall) rather than an all-zero row -- a
different failure mode than the other notebooks, not the absence of one.

**Why this is the "complicated" one -- computational cost.** Dynp's DP fill is O(n_bkps x n^2)
with `n=6001` points/series, and unlike Pelt it has no pruning to cut that down -- the `n^2` term
is unavoidable regardless of cost function. Two deliberate mitigations vs. the other notebooks,
both adjustable:

1. **`DYNP_JUMP = 5`** (vs. `jump=1` everywhere else): subsamples candidate breakpoint positions
   from 6001 to ~1200, cutting the `n^2` term by ~25x. This trades a small amount of breakpoint
   placement precision (breakpoints snap to a 5-point grid) for tractability -- raise it further
   if a light-cost smoke test is still too slow on CARC, or lower it toward 1 if CARC time allows
   and placement precision matters more than speed.
2. **Smaller seed budgets across the board**, not just for `rbf`: `LIGHT_BUDGET` is
   `pool_size=14, n_train=10, n_test=4` (vs. 20/14/6 elsewhere) and `HEAVY_BUDGET` (`rbf`, which
   is additionally ~100-1000x slower per cost evaluation on top of the `n^2` DP cost) is
   `pool_size=6, n_train=4, n_test=2` (vs. 10/7/3 elsewhere). Reasoning: in the other notebooks
   only `rbf`'s per-call cost was the bottleneck, so only `rbf` needed a smaller budget; here the
   `n^2` DP cost hits every cost function, so every budget shrinks.

**One shared `N_BKPS_GRID`/`MIN_SIZES` across all 9 costs**, unlike the other notebooks where
each cost got an individually-tuned `PENS_<cost>` range. `pen` needed per-cost tuning because its
scale depends on each cost function's fit-error magnitude; `n_bkps` is just a breakpoint count,
so the same range (grounded in the actual ground-truth transition counts above) is a reasonable
starting point for every cost. Individual cost cells can still override `N_BKPS_GRID`/`MIN_SIZES`
if a particular cost's `on_edge` flag fires.

**Everything else is unchanged**: train/test split methodology, no inner K-fold CV, the
`HEAVY_COSTS = {'rbf'}` classification, and the per-cost markdown notes about `ar`/`mahalanobis`
carried over from the other notebooks (except the `ar`/`linear` zero-detection note, corrected
above).


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ammonyte as amt
from ammonyte.utils.metrics import evaluate_detection
from joblib import Parallel, delayed, Memory
from mpl_toolkits.mplot3d import Axes3D

## Section 1 -- Configuration

In [ ]:
DATA_PATH   = "../../noise_sensitivity_data/noise_red_snr3.npz"
TOLERANCE   = 0.5   # +/- 0.5 kyr matching tolerance, matches cv_noise_sensitivity.py
RANDOM_SEED = 42
ALGO = 'Dynp'
N_JOBS = -1   # CARC run: use all allocated cores; lower this if running locally

DYNP_JUMP = 5   # candidate-breakpoint subsampling: 6001 pts -> ~1200 candidates, ~25x
                # fewer ops in Dynp's O(n_bkps * n^2) DP fill. Raise if still too slow,
                # lower toward 1 if CARC time allows and placement precision matters more.

# rbf is ~100-1000x slower per cost evaluation than every other cost function, ON TOP
# OF the O(n^2) DP cost that hits every cost function here -- it gets the smallest
# seed budget. Edit this set to reclassify a cost function as heavy.
HEAVY_COSTS = {'rbf'}
LIGHT_BUDGET = dict(pool_size=14, n_train=10, n_test=4)
HEAVY_BUDGET = dict(pool_size=6, n_train=4, n_test=2)

MIN_SIZES = [5, 10, 20, 30, 50]   # shared default -- fewer points than the pen-based
                                  # notebooks purely for tractability; override per cost
                                  # if on_edge fires

# ground-truth transition counts for red/snr3 (all 20 seeds): min=35, max=60, mean=44.85
N_BKPS_GRID = [30, 35, 40, 45, 50, 55, 60, 65]  # shared default -- spans below/above
                                                   # the true range; override per cost
                                                   # if on_edge fires


def logrange(lo, hi, n):
    return list(np.unique(np.round(np.logspace(np.log10(lo), np.log10(hi), n), 3)))


## Section 2 -- Data and ground truth

In [3]:
def transition_times_from_states(time, states):
    """True transition times (both directions) from a state-label array.

    Mirrors the same helper in cv_noise_sensitivity.py / Noise_Sensitivity_Synthetic_Data.ipynb:
    any stadial/interstadial change counts as ground truth, regardless of direction (Melcher25's
    transitions are symmetric).
    """
    change_idx = np.where(np.diff(states) != 0)[0] + 1
    return time[change_idx]


d = np.load(DATA_PATH)
time_arr   = d['time']
db_noisy   = d['db_noisy']
state_true = d['state_true']
n_seeds    = db_noisy.shape[0]

gt_times = [transition_times_from_states(time_arr, state_true[s]) for s in range(n_seeds)]
n_transitions = [len(g) for g in gt_times]
print(f"Loaded {DATA_PATH}: {n_seeds} seeds, {len(time_arr)} points/series")
print(f"Ground-truth transitions per seed: min={min(n_transitions)}, max={max(n_transitions)}, "
      f"mean={np.mean(n_transitions):.1f}")

Loaded ../../noise_sensitivity_data/noise_red_snr3.npz: 20 seeds, 6001 points/series
Ground-truth transitions per seed: min=35, max=60, mean=44.9


## Section 3 -- Detection wrapper and per-combo scoring

`jump` defaults to 1 (no subsampling, ruptures' own default) and only shows up in a param dict
when a cost's cell explicitly passes `jumps=[...]` to `run_one_cost` -- see Section 7.

In [ ]:
from dynp_scoring import score_one


def _score_one(params, seed):
    return score_one(DATA_PATH, ALGO, TOLERANCE, params, seed)


# Warm up the cache once from the main process before any Parallel(...) dispatch --
# avoids a joblib+loky race where many worker processes simultaneously try to create
# the cache's metadata folder on its very first use and step on each other
# (FileNotFoundError writing func_code.py). This is a real grid point too (cost='l1'
# is in the grid), so it's not wasted work, and after the first-ever run it will
# itself come straight back from cache.
_score_one({'cost': 'l1', 'n_bkps': N_BKPS_GRID[0], 'min_size': MIN_SIZES[0], 'jump': DYNP_JUMP}, 0)

## Section 4 -- Train/test split utilities

Seeds are split ONCE per cost function (deterministic given `RANDOM_SEED`, so rerunning this
notebook reproduces the identical split -- and, since it's the same `RANDOM_SEED` and split logic
as `binseg_grid_red_snr3.ipynb`, the light-budget costs get the identical test seeds there too,
keeping cross-method comparison apples-to-apples). The test half is never touched until each
cost's `run_one_cost` call does its single final evaluation.

In [ ]:
def seed_pool(n_seeds_total, pool_size, random_state=RANDOM_SEED):
    """Fixed subsample of `pool_size` seed indices out of n_seeds_total."""
    rng = np.random.RandomState(random_state)
    return np.sort(rng.choice(n_seeds_total, size=pool_size, replace=False))


def outer_split(pool, n_train, n_test, random_state=RANDOM_SEED):
    """Split `pool` into disjoint train/test seed-index arrays, once."""
    rng = np.random.RandomState(random_state)
    shuffled = rng.permutation(pool)
    train_idx = np.sort(shuffled[:n_train])
    test_idx = np.sort(shuffled[n_train:n_train + n_test])
    return train_idx, test_idx


def split_for_cost(cost, n_seeds_total, random_state=RANDOM_SEED):
    budget = HEAVY_BUDGET if cost in HEAVY_COSTS else LIGHT_BUDGET
    pool = seed_pool(n_seeds_total, budget['pool_size'], random_state=random_state)
    return outer_split(pool, budget['n_train'], budget['n_test'], random_state=random_state)

## Section 5 -- Grid search (train pool) + held-out evaluation (test pool)

Shared machinery every cost's cell calls into -- nothing here is cost-specific, so there's nothing
to copy-paste (and get wrong) per cost.

In [ ]:
def grid_search_train(param_grid, train_idx):
    """Score every (param, seed in train_idx) combo, aggregate per param, pick the
    winner by mean f1 over the train pool only."""
    jobs = [(pi, params, s) for pi, params in enumerate(param_grid) for s in train_idx]
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(params, s) for pi, params, s in jobs
    )
    rows = []
    for (pi, params, s), res in zip(jobs, scored):
        row = dict(params)
        row.update(pi=pi, seed=int(s), split='train', **res)
        rows.append(row)
    train_raw = pd.DataFrame(rows)

    param_cols = list(param_grid[0].keys())
    train_summary = (train_raw.groupby(param_cols)
                      .agg(mean_precision=('precision', 'mean'), std_precision=('precision', 'std'),
                           mean_recall=('recall', 'mean'), std_recall=('recall', 'std'),
                           mean_f1=('f1', 'mean'), std_f1=('f1', 'std'),
                           mean_fp=('false_positives', 'mean'))
                      .reset_index())

    best_row = train_summary.loc[train_summary['mean_f1'].idxmax()]
    best_params = {c: best_row[c] for c in param_cols}
    return best_params, best_row, train_summary, train_raw


def evaluate_test(best_params, test_idx):
    """Apply best_params to the untouched-until-now test pool, once."""
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(best_params, s) for s in test_idx
    )
    rows = []
    for s, res in zip(test_idx, scored):
        row = dict(best_params)
        row.update(seed=int(s), split='test', **res)
        rows.append(row)
    test_raw = pd.DataFrame(rows)
    return (test_raw['precision'].mean(), test_raw['recall'].mean(),
            test_raw['f1'].mean(), test_raw['false_positives'].mean(), test_raw)


def n_bkps_sweep_test(cost, n_bkps_grid, min_size, test_idx, extra_params=None):
    """Sweep n_bkps at the winning min_size (and any other winning params, e.g. jump,
    passed via extra_params), test seeds only -- precision/recall tradeoff curve for
    this cost function (the n_bkps-analog of a PR curve; n_bkps is Dynp's sensitivity
    knob in place of pen)."""
    extra_params = extra_params or {}
    jobs = [(n_bkps, s) for n_bkps in n_bkps_grid for s in test_idx]
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(dict(cost=cost, n_bkps=n_bkps, min_size=min_size, **extra_params), s)
        for n_bkps, s in jobs
    )
    rows = []
    for (n_bkps, s), res in zip(jobs, scored):
        rows.append(dict(cost=cost, n_bkps=n_bkps, min_size=min_size, seed=int(s), split='test_sweep', **res))
    sweep_raw = pd.DataFrame(rows)
    sweep_summary = (sweep_raw.groupby('n_bkps')
                      .agg(mean_precision=('precision', 'mean'), mean_recall=('recall', 'mean'),
                           mean_f1=('f1', 'mean'), mean_fp=('false_positives', 'mean'))
                      .reset_index())
    return sweep_summary, sweep_raw


results = {}   # every cost's run_one_cost result lands here, keyed by cost name


def run_one_cost(cost, n_bkps_list, min_sizes, jumps=None):
    t0 = time.perf_counter()
    train_idx, test_idx = split_for_cost(cost, n_seeds)
    budget = 'HEAVY' if cost in HEAVY_COSTS else 'light'

    if jumps is None:
        jumps = [DYNP_JUMP]

    param_grid = [{'cost': cost, 'n_bkps': n_bkps, 'min_size': ms, 'jump': j}
                  for n_bkps in n_bkps_list for ms in min_sizes for j in jumps]

    best_params, train_row, train_summary, train_raw = grid_search_train(param_grid, train_idx)
    test_p, test_r, test_f1, test_fp, test_raw = evaluate_test(best_params, test_idx)

    sweep_extra = {'jump': best_params['jump']}
    sweep_summary, sweep_raw = n_bkps_sweep_test(cost, n_bkps_list, best_params['min_size'], test_idx,
                                                  extra_params=sweep_extra)

    on_edge = (best_params['n_bkps'] in (min(n_bkps_list), max(n_bkps_list))
               or best_params['min_size'] in (min(min_sizes), max(min_sizes)))

    conclusion_row = dict(
        cost=cost, best_n_bkps=best_params['n_bkps'], best_min_size=best_params['min_size'],
        best_jump=best_params.get('jump', DYNP_JUMP),
        train_f1=train_row['mean_f1'], train_precision=train_row['mean_precision'],
        train_recall=train_row['mean_recall'],
        test_f1=test_f1, test_precision=test_p, test_recall=test_r, test_fp=test_fp,
        n_bkps_range=f"{min(n_bkps_list)}-{max(n_bkps_list)}",
        min_size_range=f"{min(min_sizes):.0f}-{max(min_sizes):.0f}",
        on_edge=on_edge, seed_budget=budget,
        n_train_seeds=len(train_idx), n_test_seeds=len(test_idx),
        train_seeds=",".join(map(str, train_idx)), test_seeds=",".join(map(str, test_idx)),
    )

    result = dict(train_idx=train_idx, test_idx=test_idx, best_params=best_params,
                  train_summary=train_summary, train_raw=train_raw, test_raw=test_raw,
                  sweep_summary=sweep_summary, sweep_raw=sweep_raw, conclusion_row=conclusion_row)

    elapsed = time.perf_counter() - t0
    edge_note = "  <-- ON GRID EDGE" if on_edge else ""
    print(f"[{budget:5s}] {cost:12s}: n_bkps={best_params['n_bkps']:>4d} min_size={best_params['min_size']:>3.0f} "
          f"jump={best_params.get('jump', DYNP_JUMP)}  train_f1={train_row['mean_f1']:.3f}  "
          f"test_f1={test_f1:.3f}  ({elapsed:.0f}s){edge_note}")

    results[cost] = result
    return result


## Section 6 -- Plot helpers

Called right after each cost's `run_one_cost` -- immediate feedback before deciding whether to
widen that cost's range and rerun just its own cells.

In [ ]:
def plot_train_surface(cost, result):
    """3D response surface of mean_f1 over the (n_bkps, min_size) TRAIN grid -- this
    is the surface the winning combo was actually picked from."""
    summary = result['train_summary']
    best_n_bkps, best_ms = result['best_params']['n_bkps'], result['best_params']['min_size']

    if 'jump' in summary.columns:
        summary = summary[summary['jump'] == result['best_params'].get('jump', DYNP_JUMP)]

    pivot = summary.pivot(index='min_size', columns='n_bkps', values='mean_f1')
    n_bkps_vals, ms_vals = pivot.columns.values, pivot.index.values
    X, Y = np.meshgrid(n_bkps_vals, ms_vals)
    Z = pivot.values

    fig = plt.figure(figsize=(6, 5.5))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(X, Y, Z, cmap='viridis', vmin=0, vmax=np.nanmax(Z),
                     edgecolor='k', linewidth=0.2, antialiased=True)

    zi, xi = list(ms_vals).index(best_ms), list(n_bkps_vals).index(best_n_bkps)
    ax.scatter([best_n_bkps], [best_ms], [Z[zi, xi]], color='red', s=70, marker='*', zorder=10)

    on_edge = result['conclusion_row']['on_edge']
    flag = 'edge -- NOT converged' if on_edge else 'interior -- converged'
    ax.set_title(f"{cost}: train mean_f1\nbest={Z[zi, xi]:.3f} @ n_bkps={best_n_bkps:.0f}, "
                 f"min_size={best_ms:.0f}\n({flag})",
                 fontsize=9, color='#b3261e' if on_edge else '#1a1a1a')
    ax.set_xlabel('n_bkps', fontsize=8)
    ax.set_ylabel('min_size', fontsize=8)
    ax.set_zlabel('mean_f1 (train)', fontsize=8)
    ax.set_zlim(0, max(np.nanmax(Z) * 1.05, 0.05))
    ax.view_init(elev=25, azim=-60)
    fig.suptitle(f"cost='{cost}': TRAIN response surface -- this is what the winner was picked from", fontsize=10)
    plt.tight_layout()
    fig.savefig(f"dynp_grid_red_snr3_{cost}_surface.png", dpi=200, bbox_inches='tight')
    plt.show()


def plot_test_pr(cost, result):
    """Test-set precision/recall tradeoff (n_bkps swept at the winning min_size/jump)."""
    sweep, row = result['sweep_summary'], result['conclusion_row']
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(sweep['mean_recall'], sweep['mean_precision'], color='#2a78d6',
             linewidth=1.5, marker='o', markersize=4, zorder=2)
    ax.scatter([row['test_recall']], [row['test_precision']], color='#e34948',
               s=90, marker='*', zorder=5, label='selected n_bkps')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('recall')
    ax.set_ylabel('precision')
    ax.set_title(f"{cost}: TEST precision/recall tradeoff")
    ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(color='#e1e0d9', linewidth=0.6)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()


## Section 7 -- Run the grid, one cost function per cell block

`N_BKPS_GRID`/`MIN_SIZES` from Section 1 are shared defaults across all 9 costs (see intro for
why that's reasonable for `n_bkps` in a way it wasn't for `pen`) -- widen a cost's grid and
rerun if `on_edge` fires. `DYNP_JUMP` is fixed at 5 for every cost below; pass a different
`jumps=[...]` to `run_one_cost` for an individual cost if you need to sweep it.


### cost = 'ar'

NOTE: for Pelt/Binseg/BottomUp/Window, `ar` came back with zero detections across every
tested combo (pen never justified a breakpoint for this cost on this data). Dynp forces exactly
`n_bkps` breakpoints regardless, so expect non-zero but low-quality detections here instead (see
intro) -- `ar` targets autoregressive-dynamics changes, not mean level, and these transitions are
pure mean shifts in red noise.


In [ ]:
result_ar = run_one_cost('ar', N_BKPS_GRID, MIN_SIZES)


In [ ]:
plot_train_surface('ar', result_ar)
plot_test_pr('ar', result_ar)


### cost = 'l1'


In [ ]:
result_l1 = run_one_cost('l1', N_BKPS_GRID, MIN_SIZES)


In [ ]:
plot_train_surface('l1', result_l1)
plot_test_pr('l1', result_l1)


### cost = 'l2'


In [ ]:
result_l2 = run_one_cost('l2', N_BKPS_GRID, MIN_SIZES)


In [ ]:
plot_train_surface('l2', result_l2)
plot_test_pr('l2', result_l2)


### cost = 'linear'

NOTE: same correction as `ar` above -- Pelt/Binseg/BottomUp/Window returned all-zero for
`linear` (regression-relationship cost, absent from this univariate data). Dynp will still place
`n_bkps` breakpoints; expect them poorly located rather than absent.


In [ ]:
result_linear = run_one_cost('linear', N_BKPS_GRID, MIN_SIZES)


In [ ]:
plot_train_surface('linear', result_linear)
plot_test_pr('linear', result_linear)


### cost = 'normal'


In [ ]:
result_normal = run_one_cost('normal', N_BKPS_GRID, MIN_SIZES)


In [ ]:
plot_train_surface('normal', result_normal)
plot_test_pr('normal', result_normal)


### cost = 'rank'


In [ ]:
result_rank = run_one_cost('rank', N_BKPS_GRID, MIN_SIZES)


In [ ]:
plot_train_surface('rank', result_rank)
plot_test_pr('rank', result_rank)


### cost = 'rbf' (HEAVY -- ~100-1000x slower per call, smaller seed budget)


In [ ]:
result_rbf = run_one_cost('rbf', N_BKPS_GRID, MIN_SIZES)


In [ ]:
plot_train_surface('rbf', result_rbf)
plot_test_pr('rbf', result_rbf)


### cost = 'clinear'


In [ ]:
result_clinear = run_one_cost('clinear', N_BKPS_GRID, MIN_SIZES)


In [ ]:
plot_train_surface('clinear', result_clinear)
plot_test_pr('clinear', result_clinear)


### cost = 'mahalanobis'

For a univariate series this fits `metric = 1/variance(signal)` as a single global scalar
(checked in `ruptures`' `CostMl` source), so the segment cost is just L2 rescaled by a constant --
it doesn't add anything L2 doesn't already capture here. Kept in the grid anyway for comparison.


In [ ]:
result_mahalanobis = run_one_cost('mahalanobis', N_BKPS_GRID, MIN_SIZES, jumps=[10])


In [ ]:
plot_train_surface('mahalanobis', result_mahalanobis)
plot_test_pr('mahalanobis', result_mahalanobis)


## Section 8 -- Overview: all costs' test PR curves together

Run once every cost above has been run at least once (checks `results` has all 9 -- if you only
reran a subset, this still works off whatever's currently in `results` from earlier in the
kernel session).


In [ ]:
def plot_test_pr_grid(results, ncols=4):
    costs = list(results.keys())
    nrows = int(np.ceil(len(costs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 3.2 * nrows), squeeze=False)

    for idx, cost in enumerate(costs):
        ax = axes[idx // ncols][idx % ncols]
        res = results[cost]
        sweep = res['sweep_summary']
        row = res['conclusion_row']

        ax.plot(sweep['mean_recall'], sweep['mean_precision'], color='#2a78d6',
                 linewidth=1.5, marker='o', markersize=3, zorder=2)
        ax.scatter([row['test_recall']], [row['test_precision']], color='#e34948',
                   s=80, marker='*', zorder=5, label='selected n_bkps')

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_title(cost, fontsize=10)
        ax.set_xlabel('recall', fontsize=7)
        ax.set_ylabel('precision', fontsize=7)
        ax.tick_params(labelsize=6)
        ax.grid(color='#e1e0d9', linewidth=0.6, zorder=0)
        ax.set_axisbelow(True)

    for j in range(len(costs), nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')

    fig.subplots_adjust(hspace=0.7, wspace=0.5, top=0.92)
    fig.suptitle("TEST set: precision/recall tradeoff (n_bkps swept at winning min_size, test seeds only)", y=0.98)
    fig.savefig("dynp_grid_red_snr3_test_pr_grid.png", dpi=200, bbox_inches='tight')
    plt.show()


plot_test_pr_grid(results)


## Section 9 -- Save raw / summary / conclusion CSVs

In [ ]:
# raw: every scored (param, seed) row across every cost function, tagged by split --
# 'train' (full grid on the train pool, what selection used), 'test' (the single
# winning combo on the test pool), 'test_sweep' (n_bkps swept at the winning min_size
# on the test pool, for the PR-curve plots above)
raw_frames = []
for cost, result in results.items():
    raw_frames.append(result['train_raw'])
    raw_frames.append(result['test_raw'])
    raw_frames.append(result['sweep_raw'])
raw_df = pd.concat(raw_frames, ignore_index=True, sort=False)
RAW_CSV = "dynp_grid_red_snr3_raw.csv"
raw_df.to_csv(RAW_CSV, index=False)
print(f"Saved {len(raw_df)} rows to {RAW_CSV}")

# summary: the TRAIN grid summary per cost function (what selection was based on) --
# this is what the Section 7 surface plots are drawn from
summary_df = pd.concat([r['train_summary'] for r in results.values()], ignore_index=True, sort=False)
SUMMARY_CSV = "dynp_grid_red_snr3_summary.csv"
summary_df.to_csv(SUMMARY_CSV, index=False)
print(f"Saved {len(summary_df)} rows to {SUMMARY_CSV}")

# conclusion: one row per cost function -- train_f1 (selection score, optimistic)
# next to test_f1 (the honest, held-out score) so the gap between them is visible
conclusion_df = (pd.DataFrame([r['conclusion_row'] for r in results.values()])
                  .sort_values('test_f1', ascending=False).reset_index(drop=True))
CONCLUSION_CSV = "dynp_grid_red_snr3_conclusion.csv"
conclusion_df.to_csv(CONCLUSION_CSV, index=False)
print(f"Saved {len(conclusion_df)} rows to {CONCLUSION_CSV}")


In [ ]:
print("=" * 92)
print("CONCLUSION: Dynp on red/snr3 -- train/test split, optimized independently per cost function")
print("=" * 92)
for _, row in conclusion_df.iterrows():
    edge_note = "  <-- ON GRID EDGE: widen this cost's range and rerun" if row['on_edge'] else ""
    gap = row['train_f1'] - row['test_f1']
    print(f"{row['cost']:>12s} [{row['seed_budget']:5s}]: n_bkps={row['best_n_bkps']:>4.0f}  min_size={row['best_min_size']:>3.0f}  "
          f"train_f1={row['train_f1']:.3f}  test_f1={row['test_f1']:.3f}  (gap={gap:+.3f}){edge_note}")

n_edge = int(conclusion_df['on_edge'].sum())
if n_edge:
    print(f"\n{n_edge} cost function(s) did NOT converge on the train grid (optimum on the tested "
          f"boundary) -- widen those ranges and rerun before trusting their numbers.")
else:
    print(f"\nAll {len(conclusion_df)} cost functions converged (train optimum interior to their tested range).")

best_overall = conclusion_df.iloc[0]
print(f"\nBest cost function by TEST f1: '{best_overall['cost']}' "
      f"(n_bkps={best_overall['best_n_bkps']:.0f}, min_size={best_overall['best_min_size']:.0f}, "
      f"test_f1={best_overall['test_f1']:.3f})")


## Section 10 -- Re-ranking by a metric other than F1

`run_one_cost` always picks its winner by `mean_f1` (`grid_search_train`'s
`train_summary['mean_f1'].idxmax()`). But `train_summary` already has `mean_precision` and
`mean_recall` for *every* grid point, not just the F1 winner -- so "what's the best parameter
if I care about recall instead" doesn't need a regrid, just a different column to argmax over.

The helpers below re-select each cost's best params from the `train_summary` already sitting in
`results` (populated by Section 7) and fetch an honest test-set score for that alternate choice.
The only new computation is `evaluate_test` on the newly-selected params -- since `_score_one` is
joblib-cached, any `(params, seed)` combo already scored during the original grid search (same
`n_bkps`/`min_size`/`jump`, just selected for a different reason) is a cache hit, not a recompute.

**Caveat:** selecting purely by recall has the same degenerate-edge risk as the `n_bkps=65`
case above, and generally worse -- with tolerance-based matching, adding *more* breakpoints
rarely hurts recall (extra breakpoints near a true one still count as a match) even though it
tanks precision. So don't be surprised if the recall-best `n_bkps` for most costs is just
`max(N_BKPS_GRID)` again with `on_edge=True` and poor `test_precision` -- that's the metric
being maximized exactly as asked, not a bug. If you need a less degenerate answer, consider
recall at a precision floor rather than raw argmax recall.


In [ ]:
def best_row_by_metric(train_summary, param_cols, metric):
    """Same selection rule as grid_search_train's best_row, but over any
    train_summary metric column (mean_f1 / mean_recall / mean_precision) instead of
    the fixed mean_f1 grid_search_train uses."""
    best_row = train_summary.loc[train_summary[metric].idxmax()]
    return best_row, {c: best_row[c] for c in param_cols}


def conclusion_row_by_metric(cost, metric):
    """Re-select `cost`'s best params by `metric` (e.g. 'mean_recall',
    'mean_precision') instead of run_one_cost's fixed mean_f1 selection, then get
    the honest held-out test score for that alternate choice. Reuses the
    train_summary/test_idx already sitting in `results[cost]` -- no regrid."""
    r = results[cost]
    param_cols = list(r['best_params'].keys())
    train_summary = r['train_summary']
    best_row, best_params = best_row_by_metric(train_summary, param_cols, metric)
    test_p, test_r, test_f1, test_fp, test_raw = evaluate_test(best_params, r['test_idx'])

    n_bkps_list = sorted(train_summary['n_bkps'].unique())
    min_sizes = sorted(train_summary['min_size'].unique())
    on_edge = (best_params['n_bkps'] in (min(n_bkps_list), max(n_bkps_list))
               or best_params['min_size'] in (min(min_sizes), max(min_sizes)))

    return dict(
        cost=cost, select_by=metric,
        best_n_bkps=best_params['n_bkps'], best_min_size=best_params['min_size'],
        best_jump=best_params.get('jump', DYNP_JUMP),
        train_f1=best_row['mean_f1'], train_precision=best_row['mean_precision'],
        train_recall=best_row['mean_recall'],
        test_f1=test_f1, test_precision=test_p, test_recall=test_r, test_fp=test_fp,
        on_edge=on_edge,
    )


def conclusion_table_by_metric(metric, sort_by=None):
    """The Section 9 conclusion table, re-selected by `metric` for every cost
    already run (call after Section 7 -- needs `results` populated)."""
    sort_by = sort_by or f"test_{metric.replace('mean_', '')}"
    return (pd.DataFrame([conclusion_row_by_metric(cost, metric) for cost in results])
            .sort_values(sort_by, ascending=False).reset_index(drop=True))


def print_conclusion_by_metric(df, metric):
    label = metric.replace('mean_', '')
    print("=" * 92)
    print(f"CONCLUSION: Dynp on red/snr3 -- best params RE-SELECTED BY {metric} (not f1)")
    print("=" * 92)
    for _, row in df.iterrows():
        edge_note = "  <-- ON GRID EDGE" if row['on_edge'] else ""
        print(f"{row['cost']:>12s}: n_bkps={row['best_n_bkps']:>4.0f}  min_size={row['best_min_size']:>3.0f}  "
              f"test_precision={row['test_precision']:.3f}  test_recall={row['test_recall']:.3f}  "
              f"test_f1={row['test_f1']:.3f}{edge_note}")
    n_edge = int(df['on_edge'].sum())
    if n_edge:
        print(f"\n{n_edge}/{len(df)} cost function(s) picked an edge value when selecting by "
              f"{label} -- see the Section 10 caveat above before trusting these as converged optima.")


In [ ]:
# best params per cost function chosen for RECALL instead of F1
conclusion_df_recall = conclusion_table_by_metric('mean_recall')
conclusion_df_recall.to_csv("dynp_grid_red_snr3_conclusion_by_recall.csv", index=False)
print_conclusion_by_metric(conclusion_df_recall, 'mean_recall')
conclusion_df_recall


In [ ]:
# same, for PRECISION -- included for symmetry, same degenerate-edge caveat applies
# in the opposite direction (precision-argmax tends toward the smallest n_bkps that
# still fires, i.e. fewer, more confident breakpoints, possibly under-segmenting)
conclusion_df_precision = conclusion_table_by_metric('mean_precision')
conclusion_df_precision.to_csv("dynp_grid_red_snr3_conclusion_by_precision.csv", index=False)
print_conclusion_by_metric(conclusion_df_precision, 'mean_precision')
conclusion_df_precision
